In [1]:
from pathlib import Path

import numpy as np

from PIL import Image, ImageDraw, ImageFilter, ImageFont



OUT_DIR = Path("media-site/animations/scanners")

OUT_DIR.mkdir(parents=True, exist_ok=True)



SIZE = 512

FPS = 24

FRAMES = 120



CYAN = (90, 240, 255)

DIM = (50, 120, 140)

WHITE = (255, 255, 255)



rng = np.random.default_rng(42)



try:

    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 16)

    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 12)

except:

    FONT = ImageFont.load_default()

    FONT_SMALL = ImageFont.load_default()





# -------------------------------------------------

# Synthetic detections

# -------------------------------------------------

detections = []



for _ in range(24):

    x = rng.integers(80, SIZE - 80)

    y = rng.integers(80, SIZE - 80)

    s = rng.integers(8, 22)



    detections.append({

        "x": x,

        "y": y,

        "s": s,

        "phase": rng.uniform(0, 2 * np.pi),

    })





# -------------------------------------------------

# Frame renderer

# -------------------------------------------------

def render_frame(frame_idx):



    phase = 2 * np.pi * frame_idx / FRAMES



    img = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))

    draw = ImageDraw.Draw(img)



    cx = cy = SIZE // 2



    # -------------------------------------------------

    # Outer reticle

    # -------------------------------------------------

    for r, a in [(180, 40), (150, 55), (120, 70)]:

        draw.arc(

            [cx-r, cy-r, cx+r, cy+r],

            start=0,

            end=360,

            fill=(*CYAN, a),

            width=1

        )



    # -------------------------------------------------

    # Rotating sweep

    # -------------------------------------------------

    sweep_angle = phase * 180 / np.pi



    sweep = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))

    sd = ImageDraw.Draw(sweep)



    for i in range(36):

        a = int(4 + i * 2.2)



        sd.pieslice(

            [cx-210, cy-210, cx+210, cy+210],

            start=sweep_angle - i * 1.4,

            end=sweep_angle - i * 1.4 + 1.2,

            fill=(*CYAN, a)

        )



    sweep = sweep.filter(ImageFilter.GaussianBlur(10))

    img.alpha_composite(sweep)



    # -------------------------------------------------

    # Crosshair

    # -------------------------------------------------

    draw.line((cx - 220, cy, cx + 220, cy), fill=(*CYAN, 60), width=1)

    draw.line((cx, cy - 220, cx, cy + 220), fill=(*CYAN, 60), width=1)



    # -------------------------------------------------

    # Scan line

    # -------------------------------------------------

    scan_y = int((frame_idx * 4) % SIZE)



    for dy in range(-6, 7):

        alpha = int(45 - abs(dy) * 6)



        if alpha > 0:

            draw.line(

                (40, scan_y + dy, SIZE - 40, scan_y + dy),

                fill=(*CYAN, alpha),

                width=1

            )



    # -------------------------------------------------

    # Detection markers

    # -------------------------------------------------

    for idx, d in enumerate(detections):



        blink = 0.55 + 0.45 * np.sin(phase * 3 + d["phase"])

        alpha = int(40 + 180 * blink)



        x = d["x"]

        y = d["y"]

        s = d["s"]



        # brackets

        draw.line((x-s, y-s, x-s//2, y-s), fill=(*CYAN, alpha), width=2)

        draw.line((x-s, y-s, x-s, y-s//2), fill=(*CYAN, alpha), width=2)



        draw.line((x+s, y-s, x+s//2, y-s), fill=(*CYAN, alpha), width=2)

        draw.line((x+s, y-s, x+s, y-s//2), fill=(*CYAN, alpha), width=2)



        draw.line((x-s, y+s, x-s//2, y+s), fill=(*CYAN, alpha), width=2)

        draw.line((x-s, y+s, x-s, y+s//2), fill=(*CYAN, alpha), width=2)



        draw.line((x+s, y+s, x+s//2, y+s), fill=(*CYAN, alpha), width=2)

        draw.line((x+s, y+s, x+s, y+s//2), fill=(*CYAN, alpha), width=2)



        # tiny center point

        rr = 2 + int(2 * blink)



        draw.ellipse(

            [x-rr, y-rr, x+rr, y+rr],

            fill=(*WHITE, min(255, alpha + 40))

        )



        # some objects pulse

        if idx % 5 == 0:

            pulse_r = int(20 + 12 * np.sin(phase * 2 + idx))



            draw.arc(

                [x-pulse_r, y-pulse_r, x+pulse_r, y+pulse_r],

                start=0,

                end=360,

                fill=(*CYAN, 80),

                width=1

            )



    # -------------------------------------------------

    # Coordinate readout

    # -------------------------------------------------

    coord_text = (

        f"RA  {12 + 0.02*np.sin(phase):05.2f}h\n"

        f"DEC +{32 + 0.08*np.cos(phase):05.2f}°\n"

        f"Z   {0.0213 + 0.0001*np.sin(phase*2):.4f}"

    )



    draw.text(

        (28, SIZE - 92),

        coord_text,

        font=FONT_SMALL,

        fill=(*CYAN, 180)

    )



    # -------------------------------------------------

    # Catalog lock

    # -------------------------------------------------

    lock_alpha = int(150 + 80 * np.sin(phase * 4))



    draw.text(

        (SIZE - 180, 32),

        "CATALOG LOCK",

        font=FONT,

        fill=(*CYAN, lock_alpha)

    )



    draw.text(

        (SIZE - 180, 56),

        "SIMBAD / VIZIER",

        font=FONT_SMALL,

        fill=(*CYAN, 140)

    )



    # -------------------------------------------------

    # Bottom ticker

    # -------------------------------------------------

    ticker_x = int(SIZE - (frame_idx * 5) % (SIZE + 300))



    draw.text(

        (ticker_x, SIZE - 26),

        "SCANNING FIELD OBJECTS • CROSSMATCHING SOURCES • IDENTIFICATION ACTIVE",

        font=FONT_SMALL,

        fill=(*CYAN, 120)

    )



    # -------------------------------------------------

    # Glow pass

    # -------------------------------------------------

    glow = img.filter(ImageFilter.GaussianBlur(2))



    final = Image.new("RGBA", (SIZE, SIZE), (0, 0, 0, 0))

    final.alpha_composite(glow)

    final.alpha_composite(img)



    return final





# -------------------------------------------------

# Generate GIF

# -------------------------------------------------

frames = [render_frame(i) for i in range(FRAMES)]



import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "survey_scanner_overlay",

    "webm",

    FPS,

)

print(f"Saved: {saved}")


Saved: animations/scanners/survey_scanner_overlay.gif


In [2]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/loaders")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 720, 96



FPS = 24



FRAMES = 96







CYAN = (80, 235, 255)



CYAN2 = (170, 255, 255)



BG = (2, 7, 13, 255)







try:



    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 18)



    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 12)



except:



    FONT = ImageFont.load_default()



    FONT_SMALL = ImageFont.load_default()











def render_loader_frame(i):



    phase = i / FRAMES



    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    x0, y0 = 36, 42



    bar_w, bar_h = 560, 22



    seg_count = 24



    gap = 4



    seg_w = (bar_w - gap * (seg_count - 1)) / seg_count







    # outer technical frame



    d.line((24, 24, 190, 24), fill=(*CYAN, 160), width=2)



    d.line((190, 24, 214, 38), fill=(*CYAN, 160), width=2)



    d.line((214, 38, 650, 38), fill=(*CYAN, 90), width=1)







    d.rectangle(



        [x0 - 8, y0 - 8, x0 + bar_w + 8, y0 + bar_h + 8],



        outline=(*CYAN, 150),



        width=2,



    )







    # animated fill



    fill_pos = (phase * 1.35) % 1.0



    active_count = int(fill_pos * seg_count)







    for s in range(seg_count):



        sx = x0 + s * (seg_w + gap)



        sy = y0







        base_alpha = 35



        alpha = base_alpha







        if s <= active_count:



            alpha = 120







        # traveling hot segment



        hot = np.exp(-0.5 * ((s - active_count) / 2.0) ** 2)



        alpha = int(alpha + 120 * hot)







        color = CYAN2 if hot > 0.55 else CYAN







        d.rectangle(



            [sx, sy, sx + seg_w, sy + bar_h],



            fill=(*color, np.clip(alpha, 0, 255)),



        )







    # scan beam



    beam_x = x0 + fill_pos * bar_w



    for dx in range(-10, 11):



        a = int(120 - abs(dx) * 8)



        if a > 0:



            d.line(



                (beam_x + dx, y0 - 10, beam_x + dx, y0 + bar_h + 10),



                fill=(*CYAN2, a),



                width=1,



            )







    # status labels



    pct = int(fill_pos * 100)



    if pct < 18:



        status = "LINKING"



    elif pct < 46:



        status = "QUERYING"



    elif pct < 74:



        status = "CROSSMATCH"



    else:



        status = "LOCK"







    d.text((28, 6), "DATA ACQUISITION", font=FONT_SMALL, fill=(*CYAN, 160))



    d.text((612, 44), f"{pct:03d}%", font=FONT, fill=(*CYAN2, 210))



    d.text((612, 66), status, font=FONT_SMALL, fill=(*CYAN, 150))







    # bottom micro ticks



    for t in range(42):



        tx = 36 + t * 13



        a = 35 + int(70 * np.sin(i * 0.12 + t * 0.45) ** 2)



        d.line((tx, 78, tx + 5, 78), fill=(*CYAN, a), width=1)







    # flash near loop end



    if phase > 0.92:



        flash = int(180 * (phase - 0.92) / 0.08)



        d.rectangle([x0 - 8, y0 - 8, x0 + bar_w + 8, y0 + bar_h + 8],



                    outline=(*CYAN2, flash), width=3)







    glow = img.filter(ImageFilter.GaussianBlur(2.2))



    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)



    return final











frames = [render_loader_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "hud_data_loader",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/loaders/hud_data_loader.gif


In [3]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter







OUT_DIR = Path("media-site/animations/loaders")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 720, 90



FPS = 24



FRAMES = 72







CYAN = (90, 240, 255)



CYAN_HOT = (180, 255, 255)







SEGMENTS = 28







BG = (2, 7, 13, 255)











def render_frame(frame_idx):







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    x0 = 42



    y0 = 34







    seg_w = 18



    seg_h = 18



    gap = 6







    active = frame_idx % (SEGMENTS + 8)







    # frame lines



    d.line((28, 24, 180, 24), fill=(*CYAN, 160), width=2)



    d.line((180, 24, 200, 34), fill=(*CYAN, 160), width=2)



    d.line((200, 34, 640, 34), fill=(*CYAN, 80), width=1)







    # segments



    for s in range(SEGMENTS):







        x = x0 + s * (seg_w + gap)



        y = y0







        alpha = 28



        color = CYAN







        # already scanned



        if s < active:



            alpha = 120







        # current hot segment



        if s == active:



            alpha = 255



            color = CYAN_HOT







        # fading tail



        if s == active - 1:



            alpha = 180







        if s == active - 2:



            alpha = 110







        d.rectangle(



            [x, y, x + seg_w, y + seg_h],



            fill=(*color, alpha)



        )







    # percentage



    pct = int(min(active / SEGMENTS, 1.0) * 100)







    d.text(



        (620, 34),



        f"{pct:03d}%",



        fill=(*CYAN_HOT, 220)



    )







    # glow



    glow = img.filter(ImageFilter.GaussianBlur(3))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "hud_segment_loader",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/loaders/hud_segment_loader.gif


In [7]:
from pathlib import Path



from PIL import Image, ImageDraw, ImageFilter







OUT_DIR = Path("media-site/animations/loaders")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 720, 90



FPS = 24







SEGMENTS = 28



FILL_FRAMES = 64



HOLD_SECONDS = 5



HOLD_FRAMES = FPS * HOLD_SECONDS



FRAMES = FILL_FRAMES + HOLD_FRAMES







CYAN = (90, 240, 255)



CYAN_HOT = (180, 255, 255)



BG = (2, 7, 13, 255)











def render_frame(frame_idx):



    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    x0 = 42



    y0 = 34







    seg_w = 18



    seg_h = 18



    gap = 6







    if frame_idx < FILL_FRAMES:



        progress = frame_idx / (FILL_FRAMES - 1)



    else:



        progress = 1.0







    active_count = int(progress * SEGMENTS)







    # frame lines



    d.line((28, 24, 180, 24), fill=(*CYAN, 160), width=2)



    d.line((180, 24, 200, 34), fill=(*CYAN, 160), width=2)



    d.line((200, 34, 640, 34), fill=(*CYAN, 80), width=1)







    # empty segment outlines



    for s in range(SEGMENTS):



        x = x0 + s * (seg_w + gap)



        y = y0







        d.rectangle(



            [x, y, x + seg_w, y + seg_h],



            outline=(*CYAN, 55),



            width=1,



        )







    # filled segments



    for s in range(active_count):



        x = x0 + s * (seg_w + gap)



        y = y0







        is_last = s == active_count - 1 and progress < 1.0







        color = CYAN_HOT if is_last else CYAN



        alpha = 255 if is_last else 145







        d.rectangle(



            [x + 2, y + 2, x + seg_w - 2, y + seg_h - 2],



            fill=(*color, alpha),



        )







    pct = int(progress * 100)







    d.text(



        (690, 18),



        f"{pct:03d}%",



        fill=(*CYAN_HOT, 220),



    )







    glow = img.filter(ImageFilter.GaussianBlur(3))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "hud_segment_loader_fill_hold",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/loaders/hud_segment_loader_fill_hold.gif


In [9]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/telemetry")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 900, 90



FPS = 24



FRAMES = 144







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



GREEN = (90, 255, 180)



YELLOW = (255, 220, 90)



PURPLE = (190, 120, 255)







SCALE = 2







W, H = 900, 90







RW = W * SCALE



RH = H * SCALE







try:



    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 18)



    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 12)



except:



    FONT = ImageFont.load_default()



    FONT_SMALL = ImageFont.load_default()











RIBBONS = [



    ("FLUX", CYAN),



    ("FIELD", PURPLE),



    ("S/N", GREEN),



    ("PHASE", YELLOW),



]











def waveform(x, phase, mode):



    if mode == 0:



        return (



            0.55 * np.sin(x * 0.022 + phase)



            + 0.28 * np.sin(x * 0.051 + phase * 1.8)



        )







    if mode == 1:



        return (



            0.40 * np.sin(x * 0.012 + phase)



            + 0.15 * np.sin(x * 0.14 + phase * 0.7)



        )







    if mode == 2:



        return (



            0.22 * np.sin(x * 0.08 + phase)



            + 0.55 * np.sign(np.sin(x * 0.03 + phase))



        )







    return (



        0.45 * np.sin(x * 0.018 + phase)



        + 0.20 * np.cos(x * 0.09 + phase * 1.2)



    )











def render_frame(frame_idx):







    phase = frame_idx / FRAMES * 2 * np.pi







    img = Image.new("RGBA", (RW, RH), BG)



    d = ImageDraw.Draw(img)







    ribbon_h = 16



    gap = 4







    for idx, (label, color) in enumerate(RIBBONS):







        y0 = 8 + idx * (ribbon_h + gap)



        cy = y0 + ribbon_h // 2







        # left label



        d.text(



            (18, y0 - 1),



            label,



            font=FONT_SMALL,



            fill=(*color, 190)



        )







        # baseline



        d.line(



            (90, cy, W - 28, cy),



            fill=(*color, 35),



            width=1



        )







        # waveform



        pts = []







        for x in range(92, W - 32, 2):







            wx = x - 92







            yy = waveform(



                wx,



                phase * (1 + idx * 0.35),



                idx



            )







            amp = 5 + idx







            y = cy + yy * amp







            # occasional spikes



            if idx == 0:



                y += (



                    8



                    * np.exp(



                        -0.5



                        * ((wx - (frame_idx * 8 % 700)) / 12) ** 2



                    )



                )







            pts.append((x, y))







        # glow pass



        for p1, p2 in zip(pts[:-1], pts[1:]):



            d.line(



                [p1, p2],



                fill=(*color, 70),



                width=3



            )







        # main line



        for p1, p2 in zip(pts[:-1], pts[1:]):



            d.line(



                [p1, p2],



                fill=(*color, 220),



                width=1



            )







        # moving scan dot



        dot_x = 92 + int((frame_idx * (4 + idx)) % (W - 130))







        dot_phase = waveform(



            dot_x,



            phase * (1 + idx * 0.35),



            idx



        )







        dot_y = cy + dot_phase * (5 + idx)







        rr = 3







        d.ellipse(



            [



                dot_x - rr,



                dot_y - rr,



                dot_x + rr,



                dot_y + rr,



            ],



            fill=(*CYAN2, 255)



        )







        # right numeric readout



        val = (



            0.5



            + 0.5



            * np.sin(phase * (1.7 + idx * 0.3))



        )







        readout = f"{val*100:05.1f}"







        d.text(



            (W - 82, y0 - 1),



            readout,



            font=FONT_SMALL,



            fill=(*color, 170)



        )







    # top technical line



    d.line(



        (0, 0, W, 0),



        fill=(*CYAN, 80),



        width=1



    )







    # subtle HUD corners



    d.line((0, 0, 40, 0), fill=(*CYAN, 180), width=2)



    d.line((0, 0, 0, 16), fill=(*CYAN, 180), width=2)







    d.line((W-40, 0, W, 0), fill=(*CYAN, 180), width=2)



    d.line((W-1, 0, W-1, 16), fill=(*CYAN, 180), width=2)







    glow = img.filter(ImageFilter.GaussianBlur(2.5))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "telemetry_ribbons",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/telemetry/telemetry_ribbons.gif


In [10]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/targeting")



OUT_DIR.mkdir(parents=True, exist_ok=True)







SIZE = 520



FPS = 24



FRAMES = 150







BG = (2, 7, 13, 255)



CYAN = (90, 240, 255)



CYAN_HOT = (190, 255, 255)



WHITE = (255, 255, 255)







try:



    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 17)



    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 12)



except:



    FONT = ImageFont.load_default()



    FONT_SMALL = ImageFont.load_default()











def status_for_frame(i):



    if i < 35:



        return "SEARCHING"



    if i < 70:



        return "ACQUIRING"



    if i < 105:



        return "STABILIZING"



    if i < 125:



        return "LOCKED"



    return "TRACKING"











def render_frame(i):



    phase = 2 * np.pi * i / FRAMES



    img = Image.new("RGBA", (SIZE, SIZE), BG)



    d = ImageDraw.Draw(img)







    cx = cy = SIZE // 2







    status = status_for_frame(i)







    # acquisition progress



    t = min(1.0, i / 120)







    # jitter dies out as lock stabilizes



    jitter_amp = (1.0 - t) * 18



    jx = np.sin(phase * 9.0) * jitter_amp



    jy = np.cos(phase * 7.0) * jitter_amp







    tx = cx + jx



    ty = cy + jy







    # shrinking lock box



    box = 190 - 80 * t + 5 * np.sin(phase * 4)



    corner = 42







    alpha = int(80 + 150 * t)







    # rotating outer rings



    for r, a in [(205, 45), (170, 65), (132, 80)]:



        start = (i * 2 + r) % 360



        d.arc(



            [cx-r, cy-r, cx+r, cy+r],



            start=start,



            end=start + 260,



            fill=(*CYAN, a),



            width=1,



        )







    # scan sweeps



    scan_x = int((i * 6) % SIZE)



    for dx in range(-5, 6):



        a = 55 - abs(dx) * 7



        if a > 0:



            d.line(



                [scan_x + dx, 45, scan_x + dx, SIZE - 45],



                fill=(*CYAN, a),



                width=1,



            )







    scan_y = int((i * 4) % SIZE)



    d.line([50, scan_y, SIZE - 50, scan_y], fill=(*CYAN, 40), width=1)







    # crosshair



    d.line([cx - 215, cy, cx - 32, cy], fill=(*CYAN, 80), width=1)



    d.line([cx + 32, cy, cx + 215, cy], fill=(*CYAN, 80), width=1)



    d.line([cx, cy - 215, cx, cy - 32], fill=(*CYAN, 80), width=1)



    d.line([cx, cy + 32, cx, cy + 215], fill=(*CYAN, 80), width=1)







    # lock box corners



    x0, y0 = tx - box / 2, ty - box / 2



    x1, y1 = tx + box / 2, ty + box / 2







    col = CYAN_HOT if status in ("LOCKED", "TRACKING") else CYAN







    # top-left



    d.line([x0, y0, x0 + corner, y0], fill=(*col, alpha), width=3)



    d.line([x0, y0, x0, y0 + corner], fill=(*col, alpha), width=3)







    # top-right



    d.line([x1, y0, x1 - corner, y0], fill=(*col, alpha), width=3)



    d.line([x1, y0, x1, y0 + corner], fill=(*col, alpha), width=3)







    # bottom-left



    d.line([x0, y1, x0 + corner, y1], fill=(*col, alpha), width=3)



    d.line([x0, y1, x0, y1 - corner], fill=(*col, alpha), width=3)







    # bottom-right



    d.line([x1, y1, x1 - corner, y1], fill=(*col, alpha), width=3)



    d.line([x1, y1, x1, y1 - corner], fill=(*col, alpha), width=3)







    # center target pulse



    pulse_r = 18 + 10 * np.sin(phase * 5) * (0.4 + t)



    d.ellipse(



        [cx - pulse_r, cy - pulse_r, cx + pulse_r, cy + pulse_r],



        outline=(*CYAN, int(60 + 90 * t)),



        width=2,



    )



    d.ellipse([cx-3, cy-3, cx+3, cy+3], fill=(*WHITE, 210))







    # correction vectors before lock



    if i < 105:



        for k in range(5):



            ang = phase * 1.5 + k * 1.25



            r0 = 60 + k * 20



            x = cx + np.cos(ang) * r0



            y = cy + np.sin(ang) * r0



            x2 = x + np.cos(ang + 0.8) * 22



            y2 = y + np.sin(ang + 0.8) * 22



            d.line([x, y, x2, y2], fill=(*CYAN, 90), width=1)







    # final confirmation flash



    if 105 <= i <= 125:



        flash = int(180 * np.sin((i - 105) / 20 * np.pi))



        d.rectangle([22, 22, SIZE-22, SIZE-22], outline=(*CYAN_HOT, flash), width=4)







    # status block



    d.text((28, 30), "TARGET ACQUISITION", font=FONT_SMALL, fill=(*CYAN, 160))



    d.text((28, 50), status, font=FONT, fill=(*CYAN_HOT, 220))







    # numeric telemetry



    err = max(0.0, 1.0 - t)



    d.text(



        (28, SIZE - 82),



        f"ERR X {err*np.sin(phase*3):+.4f}\n"



        f"ERR Y {err*np.cos(phase*2):+.4f}\n"



        f"LOCK  {int(t*100):03d}%",



        font=FONT_SMALL,



        fill=(*CYAN, 170),



    )







    # lock brackets micro ticks



    for k in range(12):



        a = int(40 + 80 * np.sin(phase * 4 + k) ** 2)



        x = SIZE - 155 + k * 9



        d.line([x, SIZE - 42, x + 5, SIZE - 42], fill=(*CYAN, a), width=1)







    glow = img.filter(ImageFilter.GaussianBlur(2.2))



    final = Image.new("RGBA", (SIZE, SIZE), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "target_acquisition_lock",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/targeting/target_acquisition_lock.gif


In [13]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/catalog")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 760, 300



FPS = 24



FRAMES = 140







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)







STATUS_COLORS = {



    "green": (90, 255, 170),



    "yellow": (255, 220, 90),



    "red": (255, 90, 120),



}







try:



    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 18)



    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 13)



except Exception:



    FONT = ImageFont.load_default()



    FONT_SMALL = ImageFont.load_default()







ROWS = [



    ("SIMBAD", "LOCK", "green"),



    ("VIZIER", "MATCH", "green"),



    ("GAIA DR3", "VERIFIED", "green"),



    ("ALADIN", "ONLINE", "green"),



    ("SPECTRUM", "READY", "yellow"),



    ("TESS", "NO DATA", "red"),



]











def smoothstep(t):



    t = np.clip(t, 0.0, 1.0)



    return t * t * (3 - 2 * t)











def row_alpha(frame_idx, row_idx):



    start = 12 + row_idx * 12



    return smoothstep((frame_idx - start) / 10)











def render_frame(frame_idx):



    phase = 2 * np.pi * frame_idx / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    # Header



    d.text(



        (30, 4),



        "CATALOG CROSSMATCH MATRIX",



        font=FONT_SMALL,



        fill=(*CYAN2, 180),



    )







    d.line((28, 26, 220, 26), fill=(*CYAN, 170), width=2)



    d.line((220, 26, 242, 40), fill=(*CYAN, 170), width=2)



    d.line((242, 40, W - 40, 40), fill=(*CYAN, 90), width=1)







    # Scan line



    scan_y = 70 + int((frame_idx * 3.5) % 170)







    for dy in range(-5, 6):



        alpha = 40 - abs(dy) * 6



        if alpha > 0:



            d.line(



                (26, scan_y + dy, W - 28, scan_y + dy),



                fill=(*CYAN, alpha),



                width=1,



            )







    # Rows



    start_y = 72



    row_h = 32







    for idx, (name, status, status_key) in enumerate(ROWS):



        alpha_factor = row_alpha(frame_idx, idx)







        if alpha_factor <= 0:



            continue







        y = start_y + idx * row_h







        pulse = 0.55 + 0.45 * np.sin(phase * 3 + idx)



        row_alpha_value = int(45 * alpha_factor)







        # Row outline only, no colored fill



        d.rectangle(



            [24, y - 4, W - 26, y + 22],



            outline=(*CYAN, row_alpha_value),



            width=1,



        )







        # Source name



        d.text(



            (40, y),



            name,



            font=FONT,



            fill=(*CYAN2, int(220 * alpha_factor)),



        )







        # Connection line



        d.line(



            (220, y + 10, 520, y + 10),



            fill=(*CYAN, int(85 * alpha_factor)),



            width=1,



        )







        # Moving signal pulse



        pulse_x = 220 + ((frame_idx * 7 + idx * 80) % 300)







        d.ellipse(



            [



                pulse_x - 3,



                y + 7,



                pulse_x + 3,



                y + 13,



            ],



            fill=(*CYAN2, int(230 * alpha_factor)),



        )







        # Status circle + cyan text



        status_color = STATUS_COLORS[status_key]







        circle_x = 552



        circle_y = y + 9



        circle_r = 6







        # Faint status slot



        d.line(



            (520, y + 10, 684, y + 10),



            fill=(*CYAN, int(45 * alpha_factor)),



            width=1,



        )







        # Glow circle



        glow_alpha = int((35 + 45 * pulse) * alpha_factor)







        d.ellipse(



            [



                circle_x - circle_r - 2,



                circle_y - circle_r - 2,



                circle_x + circle_r + 2,



                circle_y + circle_r + 2,



            ],



            fill=(*status_color, glow_alpha),



        )







        # Solid circle



        d.ellipse(



            [



                circle_x - circle_r,



                circle_y - circle_r,



                circle_x + circle_r,



                circle_y + circle_r,



            ],



            fill=(*status_color, int(230 * alpha_factor)),



        )







        d.text(



            (572, y + 1),



            status,



            font=FONT_SMALL,



            fill=(*CYAN2, int(210 * alpha_factor)),



        )







        # Micro indicators



        for k in range(7):



            micro_alpha = int(



                (35 + 75 * np.sin(phase * 5 + idx + k) ** 2)



                * alpha_factor



            )







            xx = 700 + k * 6







            d.line(



                (xx, y + 6, xx + 3, y + 6),



                fill=(*CYAN, micro_alpha),



                width=1,



            )







    # Footer ticker



    ticker_x = int(W - (frame_idx * 5) % (W + 420))







    d.text(



        (ticker_x, H - 24),



        "QUERYING REMOTE SERVICES • CROSSMATCHING SOURCES • VALIDATING IDENTIFIERS •",



        font=FONT_SMALL,



        fill=(*CYAN, 120),



    )







    # HUD corners



    d.line((0, 0, 36, 0), fill=(*CYAN, 180), width=2)



    d.line((0, 0, 0, 18), fill=(*CYAN, 180), width=2)







    d.line((W - 36, 0, W, 0), fill=(*CYAN, 180), width=2)



    d.line((W - 1, 0, W - 1, 18), fill=(*CYAN, 180), width=2)







    # Glow pass



    glow = img.filter(ImageFilter.GaussianBlur(2.4))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "catalog_crossmatch_matrix",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/catalog/catalog_crossmatch_matrix.gif


In [15]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/probes")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 820, 420



FPS = 24



FRAMES = 150







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



GREEN = (90, 255, 170)



YELLOW = (255, 220, 90)







try:



    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 18)



    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 13)



    FONT_TINY = ImageFont.truetype("DejaVuSansMono.ttf", 11)



except Exception:



    FONT = ImageFont.load_default()



    FONT_SMALL = ImageFont.load_default()



    FONT_TINY = ImageFont.load_default()











DATA_ROWS = [



    ("OBJECT", "WD J0205-053"),



    ("CLASS", "DAH"),



    ("TEMP", "11400 K"),



    ("LOG G", "8.1"),



    ("FIELD", "43 MG"),



    ("SOURCE", "GAIA DR3"),



]











def smoothstep(t):



    t = np.clip(t, 0.0, 1.0)



    return t * t * (3 - 2 * t)











def render_frame(i):



    phase = 2 * np.pi * i / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    # animated probe target



    target_x = 250 + 8 * np.sin(phase * 1.7)



    target_y = 220 + 6 * np.cos(phase * 1.3)







    card_x = 420



    card_y = 92



    card_w = 330



    card_h = 235







    appear = smoothstep((i - 12) / 25)



    line_appear = smoothstep((i - 35) / 25)



    card_appear = smoothstep((i - 48) / 22)







    # target rings



    for r, base_a in [(58, 55), (38, 85), (20, 120)]:



        pulse = 0.65 + 0.35 * np.sin(phase * 4 + r)



        a = int(base_a * appear * pulse)







        d.arc(



            [target_x-r, target_y-r, target_x+r, target_y+r],



            start=(i * 3 + r) % 360,



            end=((i * 3 + r) % 360) + 285,



            fill=(*CYAN, a),



            width=2,



        )







    # central dot



    rr = 4 + 2 * np.sin(phase * 5)



    d.ellipse(



        [target_x-rr, target_y-rr, target_x+rr, target_y+rr],



        fill=(*CYAN2, int(230 * appear)),



    )







    # lock brackets



    b = 48 - 10 * smoothstep(i / 70)



    c = 16







    bracket_alpha = int(190 * appear)







    for sx, sy in [(-1, -1), (1, -1), (-1, 1), (1, 1)]:



        x = target_x + sx * b



        y = target_y + sy * b







        d.line(



            [x, y, x - sx * c, y],



            fill=(*CYAN, bracket_alpha),



            width=2,



        )



        d.line(



            [x, y, x, y - sy * c],



            fill=(*CYAN, bracket_alpha),



            width=2,



        )







    # connector line target -> card



    if line_appear > 0:



        mid_x = target_x + (card_x - target_x) * line_appear



        mid_y = target_y + (card_y + 48 - target_y) * line_appear







        d.line(



            [target_x, target_y, mid_x, mid_y],



            fill=(*CYAN, int(130 * line_appear)),



            width=1,



        )







        # moving signal point



        dot_t = (i * 0.025) % 1.0



        px = target_x + (card_x - target_x) * dot_t



        py = target_y + (card_y + 48 - target_y) * dot_t







        d.ellipse(



            [px-3, py-3, px+3, py+3],



            fill=(*CYAN2, int(210 * line_appear)),



        )







    # card appears



    if card_appear > 0:



        ca = card_appear







        # card frame



        d.rectangle(



            [card_x, card_y, card_x + card_w, card_y + card_h],



            outline=(*CYAN, int(145 * ca)),



            width=2,



        )







        d.line(



            [card_x, card_y, card_x + 80, card_y],



            fill=(*CYAN2, int(220 * ca)),



            width=3,



        )



        d.line(



            [card_x + card_w - 90, card_y + card_h, card_x + card_w, card_y + card_h],



            fill=(*CYAN2, int(200 * ca)),



            width=3,



        )







        # header



        d.text(



            (card_x + 18, card_y + 14),



            "SCIENTIFIC PROBE",



            font=FONT_SMALL,



            fill=(*CYAN2, int(220 * ca)),



        )







        status_alpha = int((140 + 70 * np.sin(phase * 5)) * ca)



        d.text(



            (card_x + 215, card_y + 14),



            "LOCK",



            font=FONT_SMALL,



            fill=(*GREEN, status_alpha),



        )







        # horizontal separator



        d.line(



            [card_x + 18, card_y + 42, card_x + card_w - 18, card_y + 42],



            fill=(*CYAN, int(80 * ca)),



            width=1,



        )







        # rows typing/reveal



        for idx, (k, v) in enumerate(DATA_ROWS):



            row_start = 65 + idx * 10



            ra = smoothstep((i - row_start) / 10)







            if ra <= 0:



                continue







            y = card_y + 58 + idx * 25







            d.text(



                (card_x + 22, y),



                k,



                font=FONT_TINY,



                fill=(*CYAN, int(150 * ca * ra)),



            )







            d.text(



                (card_x + 122, y),



                v,



                font=FONT_SMALL,



                fill=(*CYAN2, int(220 * ca * ra)),



            )







        # small row activity marker



        mx = card_x + card_w - 36



        ma = int((70 + 80 * np.sin(phase * 4 + idx)) * ca * ra)







        marker_color = GREEN if idx < 5 else YELLOW







        d.ellipse(



            [mx-4, y+4, mx+4, y+12],



            fill=(*marker_color, ma),



        )



        # micro telemetry bars



        for k in range(16):



            x = card_x + 18 + k * 12



            y = card_y + card_h - 24



            a = int((35 + 100 * np.sin(phase * 4 + k) ** 2) * ca)







            d.line(



                [x, y, x + 6, y],



                fill=(*CYAN, a),



                width=1,



            )







    # drifting scan ticks around target



    for k in range(10):



        a = int(50 + 70 * np.sin(phase * 3 + k) ** 2)



        ang = phase * 0.7 + k * 0.63



        r0 = 76 + 8 * np.sin(phase + k)







        x0 = target_x + np.cos(ang) * r0



        y0 = target_y + np.sin(ang) * r0



        x1 = target_x + np.cos(ang) * (r0 + 12)



        y1 = target_y + np.sin(ang) * (r0 + 12)







        d.line([x0, y0, x1, y1], fill=(*CYAN, a), width=1)







    # glow pass



    glow = img.filter(ImageFilter.GaussianBlur(2.2))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "scientific_cursor_probe",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/probes/scientific_cursor_probe.gif


In [17]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/signals")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 320



FPS = 24



ANIM_SECONDS = 6



FRAMES = FPS * ANIM_SECONDS







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



GREEN = (90, 255, 170)



YELLOW = (255, 220, 90)







rng = np.random.default_rng(7)







try:



    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 20)



    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 13)



except Exception:



    FONT = ImageFont.load_default()



    FONT_SMALL = ImageFont.load_default()











def smoothstep(t):



    t = np.clip(t, 0.0, 1.0)



    return t * t * (3 - 2 * t)











# -------------------------------------------------



# Base signal



# -------------------------------------------------



X = np.linspace(0, 1, 1600)







noise = (



    0.08 * np.sin(2 * np.pi * X * 18)



    + 0.04 * np.sin(2 * np.pi * X * 47)



    + 0.03 * rng.normal(size=len(X))



)







signal = noise.copy()







burst = np.exp(-0.5 * ((X - 0.58) / 0.018) ** 2)



carrier = np.sin(2 * np.pi * X * 140)







signal += burst * carrier * 0.9



signal += burst * 0.55







decoded = (



    0.42 * np.sin(2 * np.pi * X * 22)



    + 0.18 * np.sin(2 * np.pi * X * 51)



)











# -------------------------------------------------



# Frame renderer



# -------------------------------------------------



def render_frame(i, mode="reveal"):







    phase = 2 * np.pi * i / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    # -------------------------------------------------



    # Animation phases



    # -------------------------------------------------



    if mode == "reveal":







        reveal = smoothstep((i - 5) / 25)



        detect = smoothstep((i - 35) / 18)



        lock = smoothstep((i - 62) / 20)



        decode = smoothstep((i - 88) / 22)







    else:



        reveal = 1.0



        detect = 1.0



        lock = 1.0



        decode = 1.0







    # -------------------------------------------------



    # Header



    # -------------------------------------------------



    d.text(



        (32, 10),



        "DEEP SPACE SIGNAL INTERCEPT",



        font=FONT_SMALL,



        fill=(*CYAN2, 190)



    )







    d.line((28, 34, 240, 34), fill=(*CYAN, 180), width=2)



    d.line((240, 34, 262, 48), fill=(*CYAN, 180), width=2)



    d.line((262, 48, W - 40, 48), fill=(*CYAN, 90), width=1)







    # -------------------------------------------------



    # Grid



    # -------------------------------------------------



    for gx in range(80, W - 40, 80):



        d.line((gx, 70, gx, H - 60), fill=(*CYAN, 18), width=1)







    for gy in range(80, H - 50, 40):



        d.line((40, gy, W - 40, gy), fill=(*CYAN, 18), width=1)







    # -------------------------------------------------



    # Main waveform



    # -------------------------------------------------



    cx0 = 60



    cy0 = H // 2 + 8







    scale_x = W - 120



    scale_y = 82







    final_signal = (



        signal * (1.0 - decode)



        + decoded * decode



    )







    pts = []







    for idx, x in enumerate(X):







        px = cx0 + x * scale_x







        if px > cx0 + reveal * scale_x:



            break







        yy = final_signal[idx]







        jitter = (



            (1.0 - lock)



            * 0.06



            * np.sin(idx * 0.12 + phase * 8)



        )







        py = cy0 - (yy + jitter) * scale_y







        pts.append((px, py))







    # glow



    for p1, p2 in zip(pts[:-1], pts[1:]):



        d.line([p1, p2], fill=(*CYAN, 60), width=4)







    # main line



    for p1, p2 in zip(pts[:-1], pts[1:]):



        d.line([p1, p2], fill=(*CYAN2, 220), width=1)







    # -------------------------------------------------



    # Moving scan beam



    # -------------------------------------------------



    beam_x = cx0 + int((i * 7) % scale_x)







    for dx in range(-12, 13):







        a = int(70 - abs(dx) * 5)







        if a > 0:



            d.line(



                (beam_x + dx, 70, beam_x + dx, H - 60),



                fill=(*CYAN, a),



                width=1



            )







    # -------------------------------------------------



    # Burst detection marker



    # -------------------------------------------------



    burst_x = cx0 + 0.58 * scale_x







    if detect > 0:







        pulse = 0.5 + 0.5 * np.sin(phase * 6)







        r = 24 + 6 * pulse







        d.ellipse(



            [



                burst_x - r,



                cy0 - r,



                burst_x + r,



                cy0 + r,



            ],



            outline=(*YELLOW, int(180 * detect)),



            width=2,



        )







        d.text(



            (burst_x + 34, cy0 - 20),



            "ANOMALOUS SIGNAL",



            font=FONT_SMALL,



            fill=(*YELLOW, int(220 * detect)),



        )







    # -------------------------------------------------



    # Status



    # -------------------------------------------------



    if mode == "idle":







        status = "TRACKING"



        scol = GREEN







    else:







        if i < 35:



            status = "SEARCHING"



            scol = CYAN







        elif i < 62:



            status = "DETECTING"



            scol = YELLOW







        elif i < 88:



            status = "LOCKING"



            scol = YELLOW







        else:



            status = "DECODED"



            scol = GREEN







    pulse_a = int(140 + 80 * np.sin(phase * 5))







    d.text(



        (40, H - 42),



        status,



        font=FONT,



        fill=(*scol, pulse_a),



    )







    # -------------------------------------------------



    # Telemetry



    # -------------------------------------------------



    snr = 2 + 28 * detect + 15 * decode



    conf = 100 * lock







    telemetry = [



        f"SNR      {snr:05.2f}",



        f"LOCK     {conf:05.1f}%",



        f"DRIFT    {0.002*np.sin(phase):+.4f}",



        f"CHANNEL  XB-17",



    ]







    for idx, txt in enumerate(telemetry):







        d.text(



            (W - 220, 82 + idx * 22),



            txt,



            font=FONT_SMALL,



            fill=(*CYAN, 170),



        )







    # -------------------------------------------------



    # Decode progress bar



    # -------------------------------------------------



    bx = W - 240



    by = H - 48



    bw = 180



    bh = 14







    d.rectangle(



        [bx, by, bx + bw, by + bh],



        outline=(*CYAN, 120),



        width=1,



    )







    fill_w = int((bw - 4) * decode)







    d.rectangle(



        [bx + 2, by + 2, bx + 2 + fill_w, by + bh - 2],



        fill=(*GREEN, 180),



    )







    d.text(



        (bx, by - 18),



        "DECODE",



        font=FONT_SMALL,



        fill=(*CYAN, 150),



    )







    # -------------------------------------------------



    # Footer ticker



    # -------------------------------------------------



    ticker_x = int(W - (i * 6) % (W + 500))







    d.text(



        (ticker_x, H - 18),



        "ANALYZING DEEP SPACE TRANSMISSION • FOURIER DECOMPOSITION • SIGNAL RECONSTRUCTION •",



        font=FONT_SMALL,



        fill=(*CYAN, 110),



    )







    # -------------------------------------------------



    # HUD corners



    # -------------------------------------------------



    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)



    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)







    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)



    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)







    # -------------------------------------------------



    # Glow



    # -------------------------------------------------



    glow = img.filter(ImageFilter.GaussianBlur(2.5))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











# -------------------------------------------------



# Save helper

def save_signal_anim(mode, animation_name):

    frames = [render_frame(i, mode=mode) for i in range(FRAMES)]

    import numpy as np

    from vizlib.animation_export import export_animation

    saved = export_animation(

        [np.asarray(frame.convert("RGBA")) for frame in frames],

        OUT_DIR,

        animation_name,

        "webm",

        FPS,

    )

    print(f"Saved: {saved}")





save_signal_anim("reveal", "signal_intercept_reveal")

save_signal_anim("idle", "signal_intercept_idle")

save_signal_anim("reveal", "signal_intercept")



print("Done.")



Saved: animations/signals/signal_intercept_reveal.gif
Saved: animations/signals/signal_intercept_idle.gif
Done.


In [20]:
from pathlib import Path







import numpy as np







from PIL import Image, ImageDraw, ImageFilter, ImageFont















OUT_DIR = Path("media-site/animations/signals")







OUT_DIR.mkdir(parents=True, exist_ok=True)















W, H = 960, 320







FPS = 24







ANIM_SECONDS = 6







FRAMES = FPS * ANIM_SECONDS















BG = (2, 7, 13, 255)















CYAN = (90, 240, 255)







CYAN2 = (180, 255, 255)







GREEN = (90, 255, 170)







YELLOW = (255, 220, 90)















rng = np.random.default_rng(11)















try:







    FONT = ImageFont.truetype("DejaVuSansMono.ttf", 20)







    FONT_SMALL = ImageFont.truetype("DejaVuSansMono.ttf", 13)







except Exception:







    FONT = ImageFont.load_default()







    FONT_SMALL = ImageFont.load_default()























def smoothstep(t):







    t = np.clip(t, 0.0, 1.0)







    return t * t * (3 - 2 * t)























X = np.linspace(0, 1, 1600)















base_noise = (







    0.08 * np.sin(2 * np.pi * X * 18)







    + 0.04 * np.sin(2 * np.pi * X * 47)







    + 0.025 * rng.normal(size=len(X))







)















burst = np.exp(-0.5 * ((X - 0.58) / 0.018) ** 2)







carrier = np.sin(2 * np.pi * X * 140)















raw_signal = base_noise + burst * carrier * 0.9 + burst * 0.55















decoded_signal = (







    0.42 * np.sin(2 * np.pi * X * 22)







    + 0.18 * np.sin(2 * np.pi * X * 51)







)























def live_modulation(x, phase, strength=0.22):







    return (







        1.0







        + strength * np.sin(2 * np.pi * x * 1.7 + phase * 1.3)







        + 0.10 * np.sin(2 * np.pi * x * 4.1 - phase * 0.8)







    )















def burst_envelope(t):







    # controlled burst in middle of animation







    return np.exp(







        -0.5 * ((t - 0.55) / 0.075) ** 2







    )























def make_live_signal(phase, mode, reveal_t):















    amp = live_modulation(







        X,







        phase,







        strength=0.18







    )















    calm = base_noise * amp















    if mode == "reveal":















        b = burst_envelope(reveal_t)















        burst_part = (







            burst * carrier * (0.62 * b)







            + burst * (0.34 * b)







        )















        sig = calm + burst_part















        sig = np.clip(sig, -0.82, 0.82)















        return sig















    idle = decoded_signal * live_modulation(







        X,







        phase + 1.2,







        strength=0.20







    )















    idle = np.clip(idle, -0.72, 0.72)















    return idle































def render_frame(i, mode="reveal"):







    phase = 2 * np.pi * i / FRAMES















    img = Image.new("RGBA", (W, H), BG)







    d = ImageDraw.Draw(img)















    if mode == "reveal":







        reveal = smoothstep((i - 5) / 25)







        detect = smoothstep((i - 35) / 18)







        lock = smoothstep((i - 62) / 20)







        decode = smoothstep((i - 88) / 22)







    else:







        reveal = 1.0







        detect = 1.0







        lock = 1.0







        decode = 1.0















    d.text((32, 10), "DEEP SPACE SIGNAL INTERCEPT", font=FONT_SMALL, fill=(*CYAN2, 190))







    d.line((28, 34, 240, 34), fill=(*CYAN, 180), width=2)







    d.line((240, 34, 262, 48), fill=(*CYAN, 180), width=2)







    d.line((262, 48, W - 40, 48), fill=(*CYAN, 90), width=1)















    for gx in range(80, W - 40, 80):







        d.line((gx, 70, gx, H - 60), fill=(*CYAN, 18), width=1)















    for gy in range(80, H - 50, 40):







        d.line((40, gy, W - 40, gy), fill=(*CYAN, 18), width=1)















    cx0 = 60







    cy0 = H // 2 + 8







    scale_x = W - 120







    scale_y = 82















    reveal_t = i / (FRAMES - 1)















    final_signal = make_live_signal(







        phase=phase,







        mode=mode,







        reveal_t=reveal_t,







    )























    pts = []















    for idx, x in enumerate(X):







        px = cx0 + x * scale_x















        if px > cx0 + reveal * scale_x:







            break















        yy = final_signal[idx]















        jitter = (1.0 - lock) * 0.06 * np.sin(idx * 0.12 + phase * 8)







        py = cy0 - (yy + jitter) * scale_y















        pts.append((px, py))















    for p1, p2 in zip(pts[:-1], pts[1:]):







        d.line([p1, p2], fill=(*CYAN, 60), width=4)















    for p1, p2 in zip(pts[:-1], pts[1:]):







        d.line([p1, p2], fill=(*CYAN2, 220), width=1)















    # NOTE: blue vertical scan strip intentionally removed.















    burst_x = cx0 + 0.58 * scale_x















    if detect > 0:







        pulse = 0.5 + 0.5 * np.sin(phase * 6)







        r = 24 + 6 * pulse















        d.ellipse(







            [burst_x - r, cy0 - r, burst_x + r, cy0 + r],







            outline=(*YELLOW, int(180 * detect)),







            width=2,







        )















        d.text(







            (burst_x + 34, cy0 - 20),







            "ANOMALOUS SIGNAL",







            font=FONT_SMALL,







            fill=(*YELLOW, int(220 * detect)),







        )















    if mode == "idle":







        status = "TRACKING"







        scol = GREEN







    else:







        if i < 35:







            status = "SEARCHING"







            scol = CYAN







        elif i < 62:







            status = "DETECTING"







            scol = YELLOW







        elif i < 88:







            status = "LOCKING"







            scol = YELLOW







        else:







            status = "DECODED"







            scol = GREEN















    pulse_a = int(140 + 80 * np.sin(phase * 5))















    d.text((40, H - 42), status, font=FONT, fill=(*scol, pulse_a))















    snr = 2 + 28 * detect + 15 * decode + 3.5 * np.sin(phase * 1.7)







    conf = 100 * lock















    telemetry = [







        f"SNR      {snr:05.2f}",







        f"LOCK     {conf:05.1f}%",







        f"DRIFT    {0.002*np.sin(phase):+.4f}",







        f"CHANNEL  XB-17",







    ]















    for idx, txt in enumerate(telemetry):







        d.text(







            (W - 220, 82 + idx * 22),







            txt,







            font=FONT_SMALL,







            fill=(*CYAN, 170),







        )















    bx = W - 240







    by = H - 48







    bw = 180







    bh = 14















    d.rectangle([bx, by, bx + bw, by + bh], outline=(*CYAN, 120), width=1)















    fill_w = int((bw - 4) * decode)















    d.rectangle(







        [bx + 2, by + 2, bx + 2 + fill_w, by + bh - 2],







        fill=(*GREEN, 180),







    )















    d.text((bx, by - 18), "DECODE", font=FONT_SMALL, fill=(*CYAN, 150))















    ticker_x = int(W - (i * 6) % (W + 500))















    d.text(







        (ticker_x, H - 18),







        "ANALYZING DEEP SPACE TRANSMISSION • FOURIER DECOMPOSITION • SIGNAL RECONSTRUCTION •",







        font=FONT_SMALL,







        fill=(*CYAN, 110),







    )















    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)







    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)







    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)







    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)















    glow = img.filter(ImageFilter.GaussianBlur(2.5))















    final = Image.new("RGBA", (W, H), BG)







    final.alpha_composite(glow)







    final.alpha_composite(img)















    return final























def save_signal_gif(mode, filename):







    frames = [render_frame(i, mode=mode) for i in range(FRAMES)]















    out = OUT_DIR / filename















    frames[0].save(







        out,







        save_all=True,







        append_images=frames[1:],







        duration=int(1000 / FPS),







        loop=0,







        disposal=2,







    )















    print(f"Saved: {out}")























save_signal_gif("reveal", "signal_intercept_reveal_live.gif")







save_signal_gif("idle", "signal_intercept_idle_live.gif")















print("Done.")





Saved: animations/signals/signal_intercept_reveal_live.gif
Saved: animations/signals/signal_intercept_idle_live.gif
Done.


In [29]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/orbits")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 900, 520
FPS = 24
FRAMES = 168

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
PURPLE = (190, 120, 255)

def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]

    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass

    return ImageFont.load_default()


FONT = load_font(20)
FONT_SMALL = load_font(14)


def ellipse_point(cx, cy, a, b, theta, rot=0.0):
    x = a * np.cos(theta)
    y = b * np.sin(theta)

    xr = x * np.cos(rot) - y * np.sin(rot)
    yr = x * np.sin(rot) + y * np.cos(rot)

    return cx + xr, cy + yr


def draw_orbit(draw, cx, cy, a, b, rot, color, alpha=90, width=1):
    pts = []

    for t in np.linspace(0, 2 * np.pi, 360):
        pts.append(ellipse_point(cx, cy, a, b, t, rot))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        draw.line([p1, p2], fill=(*color, alpha), width=width)

    draw.line([pts[-1], pts[0]], fill=(*color, alpha), width=width)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    cx, cy = 330, 265

    # Header
    d.text((34, 12), "ORBITAL SOLUTION HUD", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((30, 36, 225, 36), fill=(*CYAN, 170), width=2)
    d.line((225, 36, 248, 50), fill=(*CYAN, 170), width=2)
    d.line((248, 50, W - 40, 50), fill=(*CYAN, 75), width=1)

    # Reference grid
    for r, a, w in [(70, 70, 2), (120, 58, 2), (170, 46, 2), (220, 34, 2)]:
        d.ellipse(
            [cx-r, cy-r, cx+r, cy+r],
            outline=(*CYAN, a),
            width=w
        )

    for ang in np.linspace(0, 2 * np.pi, 12, endpoint=False):
        x2 = cx + np.cos(ang + phase * 0.04) * 250
        y2 = cy + np.sin(ang + phase * 0.04) * 250
        d.line((cx, cy, x2, y2), fill=(*CYAN, 38), width=1)

    WHITE = (245, 250, 255)

    # Central body
    for r, a in [(34, 26), (25, 48), (16, 150)]:
        d.ellipse(
            [cx-r, cy-r, cx+r, cy+r],
            outline=(*WHITE, a),
            width=2
        )

    d.ellipse(
        [cx-9, cy-9, cx+9, cy+9],
        fill=(*WHITE, 245)
    )

    # Orbits
    orbits = [
        {"a": 145, "b": 72, "rot": -0.28, "color": CYAN, "speed": 1.0, "label": "TRANSFER"},
        {"a": 205, "b": 105, "rot": 0.18, "color": PURPLE, "speed": 0.63, "label": "TARGET"},
        {"a": 95, "b": 48, "rot": 0.55, "color": GREEN, "speed": 1.45, "label": "INNER"},
    ]

    for od in orbits:
        draw_orbit(d, cx, cy, od["a"], od["b"], od["rot"], od["color"], alpha=85, width=1)

    # Main spacecraft marker on transfer orbit
    od = orbits[0]
    theta = phase * od["speed"] + 0.4
    sx, sy = ellipse_point(cx, cy, od["a"], od["b"], theta, od["rot"])

    # Ghost future positions
    for k in range(1, 6):
        gt = theta + k * 0.28
        gx, gy = ellipse_point(cx, cy, od["a"], od["b"], gt, od["rot"])
        alpha = 95 - k * 13
        d.ellipse([gx-3, gy-3, gx+3, gy+3], fill=(*CYAN, alpha))

    # Spacecraft marker
    d.ellipse([sx-7, sy-7, sx+7, sy+7], fill=(*CYAN2, 245))
    d.ellipse([sx-18, sy-18, sx+18, sy+18], outline=(*CYAN, 120), width=2)

    # Velocity vector
    sx2, sy2 = ellipse_point(cx, cy, od["a"], od["b"], theta + 0.08, od["rot"])
    vx, vy = sx2 - sx, sy2 - sy
    norm = max(np.hypot(vx, vy), 1)
    vx, vy = vx / norm, vy / norm

    d.line((sx, sy, sx + vx * 44, sy + vy * 44), fill=(*GREEN, 200), width=2)

    # Periapsis marker
    px, py = ellipse_point(cx, cy, od["a"], od["b"], np.pi, od["rot"])
    d.line((px-8, py, px+8, py), fill=(*YELLOW, 180), width=2)
    d.line((px, py-8, px, py+8), fill=(*YELLOW, 180), width=2)
    d.text((px + 12, py - 8), "PERIAPSIS", font=FONT_SMALL, fill=(*YELLOW, 160))

    # Uncertainty cone
    cone_phase = 0.45 + 0.12 * np.sin(phase * 2)
    c1x, c1y = ellipse_point(cx, cy, 205, 105, theta + cone_phase, 0.18)
    c2x, c2y = ellipse_point(cx, cy, 205, 105, theta + cone_phase + 0.18, 0.18)

    d.line((sx, sy, c1x, c1y), fill=(*CYAN, 45), width=1)
    d.line((sx, sy, c2x, c2y), fill=(*CYAN, 45), width=1)

    # Right telemetry panel
    panel_x, panel_y = 610, 95
    panel_w, panel_h = 260, 315

    d.rectangle([panel_x, panel_y, panel_x + panel_w, panel_y + panel_h], outline=(*CYAN, 110), width=1)
    d.line((panel_x, panel_y, panel_x + 80, panel_y), fill=(*CYAN2, 200), width=3)

    d.text((panel_x + 16, panel_y + 16), "ORBIT FIT", font=FONT_SMALL, fill=(*CYAN2, 190))

    metrics = [
        ("SMA", f"{1.42 + 0.02*np.sin(phase):.3f} AU"),
        ("ECC", f"{0.317 + 0.004*np.cos(phase):.3f}"),
        ("INC", f"{12.4 + 0.3*np.sin(phase*0.7):.1f}°"),
        ("PER", f"{411.2 + 1.8*np.cos(phase*0.9):.1f} d"),
        ("DV",  f"{3.62 + 0.08*np.sin(phase*1.4):.2f} km/s"),
        ("FIT", f"{98.2 + 0.7*np.sin(phase*2.1):.1f}%"),
    ]

    for idx, (k, v) in enumerate(metrics):
        y = panel_y + 55 + idx * 34
        d.text((panel_x + 18, y), k, font=FONT_SMALL, fill=(*CYAN, 145))
        d.text((panel_x + 90, y), v, font=FONT, fill=(*CYAN2, 220))

        # activity ticks
        for t in range(5):
            a = int(40 + 90 * np.sin(phase * 4 + idx + t) ** 2)
            tx = panel_x + 195 + t * 7
            d.line((tx, y + 8, tx + 4, y + 8), fill=(*CYAN, a), width=1)

    # Status ribbon
    d.text((35, H - 36), "PROPAGATING ORBITAL SOLUTION • FUTURE POSITION ESTIMATES ACTIVE", font=FONT_SMALL, fill=(*CYAN, 145))

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(2.0))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "orbital_solution_hud.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/orbits/orbital_solution_hud.gif


In [30]:
from pathlib import Path
import numpy as np
from PIL import Image, ImageDraw, ImageFilter, ImageFont

OUT_DIR = Path("media-site/animations/spectroscopy")
OUT_DIR.mkdir(parents=True, exist_ok=True)

W, H = 960, 300
FPS = 24
FRAMES = 150

BG = (2, 7, 13, 255)

CYAN = (90, 240, 255)
CYAN2 = (180, 255, 255)
GREEN = (90, 255, 170)
YELLOW = (255, 220, 90)
WHITE = (245, 250, 255)

def load_font(size):
    candidates = [
        "/System/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",
        "/Library/Fonts/Arial.ttf",
        "DejaVuSansMono.ttf",
    ]
    for path in candidates:
        try:
            return ImageFont.truetype(path, size)
        except Exception:
            pass
    return ImageFont.load_default()

FONT = load_font(22)
FONT_SMALL = load_font(15)
FONT_TINY = load_font(12)

LINES = [
    {"x": 180, "tag": "Ca II", "conf": 97, "color": CYAN},
    {"x": 310, "tag": "Hβ", "conf": 94, "color": CYAN2},
    {"x": 448, "tag": "He I", "conf": 88, "color": GREEN},
    {"x": 585, "tag": "Mg", "conf": 91, "color": YELLOW},
    {"x": 705, "tag": "Fe", "conf": 83, "color": CYAN},
]


def smoothstep(t):
    t = np.clip(t, 0.0, 1.0)
    return t * t * (3 - 2 * t)


def render_frame(i):
    phase = 2 * np.pi * i / FRAMES

    img = Image.new("RGBA", (W, H), BG)
    d = ImageDraw.Draw(img)

    # Header
    d.text((30, 10), "SPECTRAL IDENTIFICATION OVERLAY", font=FONT_SMALL, fill=(*CYAN2, 190))
    d.line((28, 34, 235, 34), fill=(*CYAN, 170), width=2)
    d.line((235, 34, 258, 48), fill=(*CYAN, 170), width=2)
    d.line((258, 48, W - 40, 48), fill=(*CYAN, 80), width=1)

    # Spectrum frame
    sx0, sy0 = 70, 92
    sx1, sy1 = W - 70, 210
    mid_y = (sy0 + sy1) // 2

    d.rectangle([sx0, sy0, sx1, sy1], outline=(*CYAN, 90), width=1)

    for gx in range(sx0 + 50, sx1, 80):
        d.line((gx, sy0, gx, sy1), fill=(*CYAN, 20), width=1)

    for gy in range(sy0 + 25, sy1, 25):
        d.line((sx0, gy, sx1, gy), fill=(*CYAN, 16), width=1)

    # Synthetic spectrum
    xs = np.linspace(sx0 + 18, sx1 - 18, 850)
    wave = (
        0.20 * np.sin(xs * 0.035 + phase * 0.8)
        + 0.08 * np.sin(xs * 0.11 - phase * 1.2)
    )

    y = mid_y + wave * 22

    for item in LINES:
        dip = np.exp(-0.5 * ((xs - item["x"]) / 11) ** 2)
        y += dip * 42

    pts = list(zip(xs, y))

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN, 55), width=3)

    for p1, p2 in zip(pts[:-1], pts[1:]):
        d.line([p1, p2], fill=(*CYAN2, 225), width=1)

    # Scan line
    scan_t = (i % FRAMES) / (FRAMES - 1)
    scan_x = sx0 + scan_t * (sx1 - sx0)

    for dx in range(-6, 7):
        a = 70 - abs(dx) * 9
        if a > 0:
            d.line((scan_x + dx, sy0 - 14, scan_x + dx, sy1 + 14), fill=(*CYAN, a), width=1)

    # Markers become active after scan passes them
    for idx, item in enumerate(LINES):
        x = item["x"]
        passed = smoothstep((scan_x - x) / 38)

        if passed <= 0:
            continue

        color = item["color"]
        pulse = 0.65 + 0.35 * np.sin(phase * 5 + idx)
        a = int(220 * passed)

        # vertical lock marker
        d.line((x, sy0 - 8, x, sy1 + 8), fill=(*color, int(95 * passed)), width=1)

        # top triangle
        d.polygon(
            [(x, sy0 - 18), (x - 6, sy0 - 8), (x + 6, sy0 - 8)],
            fill=(*color, int(a * pulse)),
        )

        # bottom triangle
        d.polygon(
            [(x, sy1 + 18), (x - 6, sy1 + 8), (x + 6, sy1 + 8)],
            fill=(*color, int(a * pulse)),
        )

        # label card
        card_x = x - 35
        card_y = sy0 - 58 if idx % 2 == 0 else sy1 + 28

        d.rectangle(
            [card_x, card_y, card_x + 92, card_y + 34],
            outline=(*color, int(130 * passed)),
            width=1,
        )

        d.text((card_x + 8, card_y + 4), item["tag"], font=FONT_TINY, fill=(*WHITE, int(230 * passed)))
        d.text(
            (card_x + 8, card_y + 18),
            f"CONF {item['conf']}%",
            font=FONT_TINY,
            fill=(*color, int(200 * passed)),
        )

    # Right status stack
    px, py = 740, 224
    d.text((px, py), "LINE MATCH", font=FONT_TINY, fill=(*CYAN, 150))

    active_count = sum(1 for item in LINES if scan_x > item["x"])

    d.text((px, py + 18), f"{active_count:02d}/{len(LINES):02d}", font=FONT, fill=(*CYAN2, 220))

    # Footer
    d.text(
        (32, H - 32),
        "SCANNING ABSORPTION FEATURES • MATCHING LINE DATABASE • CONFIDENCE LOCK ACTIVE",
        font=FONT_SMALL,
        fill=(*CYAN, 130),
    )

    # HUD corners
    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)
    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)
    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)
    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)

    glow = img.filter(ImageFilter.GaussianBlur(2.2))
    final = Image.new("RGBA", (W, H), BG)
    final.alpha_composite(glow)
    final.alpha_composite(img)

    return final.convert("RGB")


frames = [render_frame(i) for i in range(FRAMES)]

OUT = OUT_DIR / "spectral_identification_overlay.gif"

frames[0].save(
    OUT,
    save_all=True,
    append_images=frames[1:],
    duration=int(1000 / FPS),
    loop=0,
    disposal=2,
)

print(f"Saved: {OUT}")

Saved: animations/spectroscopy/spectral_identification_overlay.gif


In [32]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter







OUT_DIR = Path("media-site/animations/noise")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 540



FPS = 24



FRAMES = 120







BG = (2, 7, 13, 255)



CYAN = (90, 240, 255)



WHITE = (245, 250, 255)







rng = np.random.default_rng(17)











def render_frame(i):



    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    # faint sensor grain



    for _ in range(1400):



        x = int(rng.integers(0, W))



        y = int(rng.integers(0, H))



        a = int(rng.integers(14, 58))



        img.putpixel((x, y), (*WHITE, a))







    # hot pixels



    for _ in range(65):



        x = int(rng.integers(0, W))



        y = int(rng.integers(0, H))



        a = int(rng.integers(90, 240))



        r = int(rng.integers(1, 3))



        d.rectangle([x-r, y-r, x+r, y+r], fill=(*CYAN, a))







    # rolling readout band



    band_y = int((i * 5) % H)



    for dy in range(-10, 11):



        a = max(0, 48 - abs(dy) * 3)



        d.line((0, band_y + dy, W, band_y + dy), fill=(*CYAN, a), width=1)







    # occasional cosmic ray streaks



    if i % 7 == 0:



        for _ in range(2):



            x0 = int(rng.integers(80, W - 80))



            y0 = int(rng.integers(60, H - 60))



            length = int(rng.integers(40, 130))



            angle = rng.uniform(-0.7, 0.7)







            x1 = x0 + int(np.cos(angle) * length)



            y1 = y0 + int(np.sin(angle) * length)







            d.line((x0, y0, x1, y1), fill=(*WHITE, 135), width=1)



            d.line((x0, y0 + 1, x1, y1 + 1), fill=(*CYAN, 65), width=1)







    # subtle vertical sensor columns



    for x in range(0, W, 64):



        a = int(8 + 8 * np.sin(i * 0.12 + x * 0.03) ** 2)



        d.line((x, 0, x, H), fill=(*CYAN, a), width=1)







    # global sensor flicker



    flicker = int(



        10



        + 22 * np.sin(i * 0.7) ** 2



    )







    overlay = Image.new(



        "RGBA",



        (W, H),



        (*CYAN, flicker)



    )







    img.alpha_composite(overlay)







    # very soft glow



    glow = img.filter(ImageFilter.GaussianBlur(0.8))



    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "ccd_noise_overlay",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/noise/ccd_noise_overlay.gif


In [33]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter







OUT_DIR = Path("media-site/animations/grids")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 540



FPS = 24



FRAMES = 144







BG = (2, 7, 13, 255)



CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)







GRID_STEP = 48











def warp_point(x, y, phase):



    cx = W * 0.52 + 35 * np.sin(phase * 0.7)



    cy = H * 0.50 + 28 * np.cos(phase * 0.9)







    dx = x - cx



    dy = y - cy



    r2 = dx * dx + dy * dy







    strength = 18 * np.sin(phase * 1.2)



    falloff = np.exp(-r2 / (2 * 150**2))







    wx = x + strength * falloff * dx / 120



    wy = y + strength * falloff * dy / 120







    drift_x = 4 * np.sin(phase + y * 0.01)



    drift_y = 3 * np.cos(phase * 0.8 + x * 0.01)







    return wx + drift_x, wy + drift_y











def draw_polyline(draw, points, fill, width=1):



    for p1, p2 in zip(points[:-1], points[1:]):



        draw.line([p1, p2], fill=fill, width=width)











def render_frame(i):



    phase = 2 * np.pi * i / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    # grid vertical lines



    for x in range(-GRID_STEP, W + GRID_STEP, GRID_STEP):



        pts = []



        for y in range(0, H + 1, 10):



            pts.append(warp_point(x, y, phase))







        alpha = 38 if x % (GRID_STEP * 2) else 58



        draw_polyline(d, pts, (*CYAN, alpha), width=1)







    # grid horizontal lines



    for y in range(-GRID_STEP, H + GRID_STEP, GRID_STEP):



        pts = []



        for x in range(0, W + 1, 10):



            pts.append(warp_point(x, y, phase))







        alpha = 34 if y % (GRID_STEP * 2) else 54



        draw_polyline(d, pts, (*CYAN, alpha), width=1)







    # central alignment rings



    cx = W * 0.52 + 35 * np.sin(phase * 0.7)



    cy = H * 0.50 + 28 * np.cos(phase * 0.9)







    for r, a in [(42, 70), (86, 45), (132, 28)]:



        rr = r + 5 * np.sin(phase * 2 + r)



        d.ellipse(



            [cx - rr, cy - rr, cx + rr, cy + rr],



            outline=(*CYAN2, a),



            width=1,



        )







    # transient alignment markers



    for k in range(10):



        ang = phase * 0.8 + k * 0.63



        radius = 160 + 28 * np.sin(phase * 1.3 + k)







        mx = cx + np.cos(ang) * radius



        my = cy + np.sin(ang) * radius







        a = int(45 + 80 * np.sin(phase * 3 + k) ** 2)







        d.line([mx - 10, my, mx + 10, my], fill=(*CYAN2, a), width=1)



        d.line([mx, my - 10, mx, my + 10], fill=(*CYAN2, a), width=1)







    # scanning diagonal calibration line



    offset = (i * 9) % (W + H)



    d.line(



        [offset - H, H, offset, 0],



        fill=(*CYAN2, 42),



        width=1,



    )







    # corner ticks



    for x, y, sx, sy in [



        (0, 0, 1, 1),



        (W - 1, 0, -1, 1),



        (0, H - 1, 1, -1),



        (W - 1, H - 1, -1, -1),



    ]:



        d.line([x, y, x + sx * 52, y], fill=(*CYAN, 150), width=2)



        d.line([x, y, x, y + sy * 28], fill=(*CYAN, 150), width=2)







    glow = img.filter(ImageFilter.GaussianBlur(1.2))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "adaptive_grid_overlay",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/grids/adaptive_grid_overlay.gif


In [34]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter







OUT_DIR = Path("media-site/animations/glitch")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 540



FPS = 24



FRAMES = 120







BG = (2, 7, 13, 255)



CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



RED = (255, 70, 100)



WHITE = (245, 250, 255)







rng = np.random.default_rng(31)











def glitch_strength(i):



    bursts = [18, 43, 78, 104]



    s = 0.0



    for b in bursts:



        s += np.exp(-0.5 * ((i - b) / 3.0) ** 2)



    return min(1.0, s)











def render_frame(i):



    strength = glitch_strength(i)







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    if strength <= 0.02:



        return img







    # horizontal corrupted slices



    slice_count = int(8 + 24 * strength)







    for _ in range(slice_count):



        y = int(rng.integers(20, H - 20))



        h = int(rng.integers(2, 12 + int(18 * strength)))



        x0 = int(rng.integers(-80, W // 2))



        x1 = int(rng.integers(W // 2, W + 80))



        shift = int(rng.integers(-80, 80) * strength)







        alpha = int(rng.integers(35, 150) * strength)







        color = CYAN2 if rng.random() > 0.18 else RED







        d.rectangle(



            [x0 + shift, y, x1 + shift, y + h],



            fill=(*color, alpha),



        )







    # broken scanlines



    for _ in range(int(20 + 70 * strength)):



        y = int(rng.integers(0, H))



        x0 = int(rng.integers(0, W - 120))



        length = int(rng.integers(40, 260))



        alpha = int(rng.integers(20, 120) * strength)







        d.line(



            [x0, y, min(W, x0 + length), y],



            fill=(*CYAN, alpha),



            width=1,



        )







    # block corruption



    for _ in range(int(4 + 14 * strength)):



        x = int(rng.integers(0, W - 80))



        y = int(rng.integers(0, H - 50))



        bw = int(rng.integers(20, 110))



        bh = int(rng.integers(8, 42))



        alpha = int(rng.integers(20, 95) * strength)







        d.rectangle(



            [x, y, x + bw, y + bh],



            outline=(*CYAN2, alpha),



            width=1,



        )







    # digital tearing vertical fragments



    for _ in range(int(3 + 10 * strength)):



        x = int(rng.integers(20, W - 20))



        y0 = int(rng.integers(0, H - 80))



        y1 = y0 + int(rng.integers(40, 170))



        alpha = int(rng.integers(30, 130) * strength)







        d.line(



            [x, y0, x + int(rng.integers(-12, 12)), y1],



            fill=(*WHITE, alpha),



            width=1,



        )







    # occasional warning micro text bars



    if strength > 0.35:



        for k in range(5):



            y = 40 + k * 26 + int(rng.integers(-5, 5))



            x = int(rng.integers(20, W - 300))



            alpha = int(120 * strength)







            d.rectangle(



                [x, y, x + 220, y + 14],



                outline=(*RED, alpha),



                width=1,



            )







            d.line(



                [x + 8, y + 7, x + 210, y + 7],



                fill=(*RED, alpha),



                width=1,



            )







    glow = img.filter(ImageFilter.GaussianBlur(1.6))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "data_corruption_glitch_overlay",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/glitch/data_corruption_glitch_overlay.gif


In [35]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/telemetry")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 180



FPS = 24



FRAMES = 168







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



GREEN = (90, 255, 170)



YELLOW = (255, 220, 90)







rng = np.random.default_rng(44)











def load_font(size):



    candidates = [



        "/System/Library/Fonts/Menlo.ttc",



        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",



        "/Library/Fonts/Arial.ttf",



        "DejaVuSansMono.ttf",



    ]



    for path in candidates:



        try:



            return ImageFont.truetype(path, size)



        except Exception:



            pass



    return ImageFont.load_default()











FONT = load_font(18)



FONT_SMALL = load_font(13)



FONT_TINY = load_font(11)











CHANNELS = [



    ("RA", "12h 31m 09.4s"),



    ("DEC", "+41° 23′ 18″"),



    ("UTC", "SYNC"),



    ("SNR", "LOCK"),



    ("PACKET", "STREAM"),



]











def render_wave(draw, x0, y0, w, phase, color):



    pts = []







    for x in range(w):



        xx = x0 + x



        yy = (



            y0



            + 7 * np.sin(x * 0.06 + phase)



            + 3 * np.sin(x * 0.17 - phase * 1.7)



        )



        pts.append((xx, yy))







    for p1, p2 in zip(pts[:-1], pts[1:]):



        draw.line([p1, p2], fill=(*color, 190), width=1)











def render_frame(i):



    phase = 2 * np.pi * i / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    # Outer technical rails



    d.line((18, 20, 230, 20), fill=(*CYAN, 160), width=2)



    d.line((230, 20, 250, 34), fill=(*CYAN, 160), width=2)



    d.line((250, 34, W - 24, 34), fill=(*CYAN, 75), width=1)







    d.line((18, H - 28, W - 36, H - 28), fill=(*CYAN, 55), width=1)







    d.text((24, 3), "DEEP SPACE TELEMETRY STREAM", font=FONT_TINY, fill=(*CYAN2, 170))







    # Moving packet bars



    for k in range(36):



        x = 24 + k * 13



        a = int(35 + 105 * np.sin(phase * 5 + k * 0.65) ** 2)



        d.line((x, 46, x + 6, 46), fill=(*CYAN, a), width=1)







    # Channel rows



    start_y = 62



    row_h = 22







    for idx, (name, label) in enumerate(CHANNELS):



        y = start_y + idx * row_h







        row_pulse = int(35 + 45 * np.sin(phase * 3 + idx) ** 2)







        d.rectangle(



            [22, y - 3, W - 28, y + 17],



            outline=(*CYAN, row_pulse),



            width=1,



        )







        d.text((34, y), name, font=FONT_TINY, fill=(*CYAN, 160))







        # Live values



        if name == "UTC":



            value = f"T+{(i / FPS):06.2f}s"



        elif name == "SNR":



            value = f"{42 + 8*np.sin(phase*1.7):05.2f} dB"



        elif name == "PACKET":



            value = f"{(1000 + i*7) % 9999:04d} / OK"



        else:



            value = label







        d.text((105, y), value, font=FONT_TINY, fill=(*CYAN2, 190))







        # waveform segment



        render_wave(



            d,



            275,



            y + 7,



            300,



            phase * (1 + idx * 0.2),



            CYAN,



        )







        # signal strength ticks



        ticks = 12



        level = int(4 + 8 * (0.5 + 0.5 * np.sin(phase * (1.1 + idx * 0.3))))







        for t in range(ticks):



            x = 610 + t * 10



            alpha = 170 if t < level else 35



            col = GREEN if t < level else CYAN







            d.rectangle(



                [x, y + 2, x + 5, y + 12],



                fill=(*col, alpha),



            )







        # sync dot



        dot_color = GREEN if idx != 4 else YELLOW



        dot_alpha = int(120 + 90 * np.sin(phase * 4 + idx) ** 2)







        d.ellipse(



            [760, y + 3, 770, y + 13],



            fill=(*dot_color, dot_alpha),



        )







        d.text(



            (790, y),



            "SYNC" if idx != 4 else "BUFFER",



            font=FONT_TINY,



            fill=(*CYAN, 140),



        )







    # Scrolling footer



    ticker_x = int(W - (i * 5) % (W + 520))







    d.text(



        (ticker_x, H - 20),



        "COORDINATE DRIFT NOMINAL • PACKET LOSS 0.02% • OBSERVATORY LINK STABLE •",



        font=FONT_TINY,



        fill=(*CYAN, 115),



    )







    # Scan pulse



    sx = 22 + int((i * 6) % (W - 60))



    d.line((sx, 52, sx, H - 40), fill=(*CYAN2, 45), width=1)







    # Corners



    d.line((0, 0, 40, 0), fill=(*CYAN, 180), width=2)



    d.line((0, 0, 0, 20), fill=(*CYAN, 180), width=2)



    d.line((W - 40, 0, W, 0), fill=(*CYAN, 180), width=2)



    d.line((W - 1, 0, W - 1, 20), fill=(*CYAN, 180), width=2)







    glow = img.filter(ImageFilter.GaussianBlur(1.6))







    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "deep_space_telemetry",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/telemetry/deep_space_telemetry.gif


In [37]:
from pathlib import Path

import numpy as np

from PIL import Image, ImageDraw, ImageFilter, ImageFont



OUT_DIR = Path("media-site/animations/telemetry")

OUT_DIR.mkdir(parents=True, exist_ok=True)



W, H = 384, 720

FPS = 24

FRAMES = 168



BG = (2, 7, 13, 255)



CYAN = (90, 240, 255)

CYAN2 = (180, 255, 255)

GREEN = (90, 255, 170)

YELLOW = (255, 220, 90)

PURPLE = (190, 120, 255)



def load_font(size):

    candidates = [

        "/System/Library/Fonts/Menlo.ttc",

        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",

        "/Library/Fonts/Arial.ttf",

        "DejaVuSansMono.ttf",

    ]

    for path in candidates:

        try:

            return ImageFont.truetype(path, size)

        except Exception:

            pass

    return ImageFont.load_default()



FONT_TINY = load_font(11)



CHANNELS = [

    ("RA", "12h31m09.4s"),

    ("DEC", "+41°23′18″"),

    ("UTC", "SYNC"),

    ("SNR", "LOCK"),

    ("PACKET", "STREAM"),

    ("FLUX", "NOMINAL"),

    ("FIELD", "STABLE"),

    ("TEMP", "LOW"),

    ("PHASE", "0.742"),

    ("DRIFT", "+0.002"),

    ("GUIDE", "ACTIVE"),

    ("BUFFER", "OK"),

]



def render_wave(draw, x0, y0, w, phase, color):

    pts = []

    for x in range(w):

        xx = x0 + x

        yy = y0 + 5 * np.sin(x * 0.08 + phase) + 2 * np.sin(x * 0.23 - phase * 1.7)

        pts.append((xx, yy))



    for p1, p2 in zip(pts[:-1], pts[1:]):

        draw.line([p1, p2], fill=(*color, 185), width=1)



def render_frame(i):

    phase = 2 * np.pi * i / FRAMES



    img = Image.new("RGBA", (W, H), BG)

    d = ImageDraw.Draw(img)



    d.text((18, 8), "DEEP SPACE TELEMETRY", font=FONT_TINY, fill=(*CYAN2, 175))

    d.line((16, 28, 138, 28), fill=(*CYAN, 160), width=2)

    d.line((138, 28, 154, 40), fill=(*CYAN, 160), width=2)

    d.line((154, 40, W - 18, 40), fill=(*CYAN, 70), width=1)



    # top packet ticks

    for k in range(22):

        x = 18 + k * 10

        a = int(35 + 105 * np.sin(phase * 5 + k * 0.65) ** 2)

        d.line((x, 54, x + 5, 54), fill=(*CYAN, a), width=1)



    start_y = 76

    row_h = 46



    for idx, (name, label) in enumerate(CHANNELS):

        y = start_y + idx * row_h

        if y > H - 54:

            break



        row_alpha = int(30 + 45 * np.sin(phase * 3 + idx) ** 2)



        d.rectangle(

            [16, y - 5, W - 12, y + 35],

            outline=(*CYAN, row_alpha),

            width=1,

        )



        d.text((26, y), name, font=FONT_TINY, fill=(*CYAN, 165))



        if name == "UTC":

            value = f"T+{(i / FPS):05.1f}s"

        elif name == "SNR":

            value = f"{42 + 8*np.sin(phase*1.7):04.1f}dB"

        elif name == "PACKET":

            value = f"{(1000 + i*7) % 9999:04d}"

        elif name == "FLUX":

            value = f"{0.72 + 0.08*np.sin(phase*1.3):.3f}"

        elif name == "FIELD":

            value = f"{43 + 2*np.sin(phase*1.9):04.1f}MG"

        elif name == "TEMP":

            value = f"{11400 + 90*np.sin(phase):05.0f}K"

        elif name == "PHASE":

            value = f"{(0.742 + 0.015*np.sin(phase)):.3f}"

        elif name == "DRIFT":

            value = f"{0.002*np.sin(phase):+.4f}"

        elif name == "GUIDE":

            value = "ACTIVE"

        elif name == "BUFFER":

            value = f"{76 + 12*np.sin(phase*1.5):04.1f}%"

        else:

            value = label



        d.text((92, y), value, font=FONT_TINY, fill=(*CYAN2, 190))



        render_wave(

            d,

            24,

            y + 25,

            W - 110,

            phase * (1 + idx * 0.16),

            CYAN,

        )



        # compact signal ticks

        ticks = 7

        level = int(2 + 5 * (0.5 + 0.5 * np.sin(phase * (1.1 + idx * 0.3))))



        for t in range(ticks):

            x = W - 86 + t * 8

            alpha = 175 if t < level else 35

            col = GREEN if t < level else CYAN

            d.rectangle([x, y + 8, x + 4, y + 20], fill=(*col, alpha))



        # sync dot

        dot_color = GREEN if idx % 4 != 0 else YELLOW

        dot_alpha = int(120 + 90 * np.sin(phase * 4 + idx) ** 2)

        d.ellipse([W - 28, y + 10, W - 18, y + 20], fill=(*dot_color, dot_alpha))



    # vertical scan pulse

    sy = 62 + int((i * 5) % (H - 120))

    d.line((10, sy, W - 10, sy), fill=(*CYAN2, 38), width=1)



    # footer

    d.text(

        (18, H - 26),

        "PACKET LOSS 0.02% • LINK STABLE",

        font=FONT_TINY,

        fill=(*CYAN, 120),

    )



    # corners

    d.line((0, 0, 34, 0), fill=(*CYAN, 180), width=2)

    d.line((0, 0, 0, 18), fill=(*CYAN, 180), width=2)

    d.line((W - 34, 0, W, 0), fill=(*CYAN, 180), width=2)

    d.line((W - 1, 0, W - 1, 18), fill=(*CYAN, 180), width=2)



    glow = img.filter(ImageFilter.GaussianBlur(1.4))



    final = Image.new("RGBA", (W, H), BG)

    final.alpha_composite(glow)

    final.alpha_composite(img)



    return final



frames = [render_frame(i) for i in range(FRAMES)]



OUT = OUT_DIR / "deep_space_telemetry_vertical.gif"

W - 18

frames[0].save(

    OUT,

    save_all=True,

    append_images=frames[1:],

    duration=int(1000 / FPS),

    loop=0,

    disposal=2,

    transparency=0,

)



print(f"Saved: {OUT}")


Saved: animations/telemetry/deep_space_telemetry_vertical.gif


In [44]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/tracking")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 540



FPS = 24



FRAMES = 168







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



GREEN = (90, 255, 170)



YELLOW = (255, 220, 90)



RED = (255, 90, 120)



LIGHT_GREY = (210, 220, 230)



DARK_GREY = (90, 100, 110)







def load_font(size):



    candidates = [



        "/System/Library/Fonts/Menlo.ttc",



        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",



        "/Library/Fonts/Arial.ttf",



        "DejaVuSansMono.ttf",



    ]



    for path in candidates:



        try:



            return ImageFont.truetype(path, size)



        except Exception:



            pass



    return ImageFont.load_default()







FONT = load_font(28)



FONT_SMALL = load_font(16)



FONT_TINY = load_font(13)











def bezier(p0, p1, p2, p3, t):



    return (



        (1 - t) ** 3 * p0



        + 3 * (1 - t) ** 2 * t * p1



        + 3 * (1 - t) * t ** 2 * p2



        + t ** 3 * p3



    )











P0 = np.array([110, 390])



P1 = np.array([310, 120])



P2 = np.array([650, 470])



P3 = np.array([850, 170])







TRACK = np.array([



    bezier(P0, P1, P2, P3, t)



    for t in np.linspace(0, 1, 420)



])











def render_frame(i):



    phase = 2 * np.pi * i / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    # Header



    d.text((32, 10), "TARGET PREDICTION HUD", font=FONT_SMALL, fill=(*CYAN2, 190))



    d.line((28, 34, 220, 34), fill=(*CYAN, 170), width=2)



    d.line((220, 34, 242, 48), fill=(*CYAN, 170), width=2)



    d.line((242, 48, W - 42, 48), fill=(*CYAN, 75), width=1)







    # Sparse background grid



    for x in range(80, W, 80):



        d.line((x, 72, x, H - 54), fill=(*CYAN, 16), width=1)







    for y in range(90, H - 50, 60):



        d.line((42, y, W - 42, y), fill=(*CYAN, 16), width=1)







    # Current object index



    idx = int((i / FRAMES) * (len(TRACK) - 90))



    idx = max(0, min(idx, len(TRACK) - 90))







    obj = TRACK[idx]







    # Historical track



    trail_start = max(0, idx - 80)



    trail = TRACK[trail_start:idx + 1]







    for n, (p1, p2) in enumerate(zip(trail[:-1], trail[1:])):



        a = int(25 + 90 * n / max(1, len(trail)))



        d.line([tuple(p1), tuple(p2)], fill=(*CYAN, a), width=1)







    # Future path



    future = TRACK[idx:idx + 90]







    for n, (p1, p2) in enumerate(zip(future[:-1], future[1:])):



        a = int(110 - 85 * n / max(1, len(future)))



        d.line([tuple(p1), tuple(p2)], fill=(*YELLOW, a), width=2 if n < 35 else 1)







    # Future ghost positions



    for k in range(1, 8):



        gi = min(idx + k * 12, len(TRACK) - 1)



        gx, gy = TRACK[gi]







        a = 150 - k * 16



        r = 5 + k * 0.4







        d.ellipse([gx-r, gy-r, gx+r, gy+r], outline=(*YELLOW, a), width=1)







        d.text(



            (gx + 8, gy - 8),



            f"+{k*3}s",



            font=FONT_TINY,



            fill=(*YELLOW, max(40, a)),



        )







    # Uncertainty cone



    cone_tip = obj



    cone_end = TRACK[min(idx + 78, len(TRACK) - 1)]







    vx = cone_end[0] - cone_tip[0]



    vy = cone_end[1] - cone_tip[1]



    norm = max(np.hypot(vx, vy), 1)







    vx, vy = vx / norm, vy / norm



    px, py = -vy, vx







    spread = 34 + 10 * np.sin(phase * 2.0)



    c1 = cone_end + np.array([px, py]) * spread



    c2 = cone_end - np.array([px, py]) * spread







    cone_layer = Image.new("RGBA", (W, H), (0, 0, 0, 0))



    cd = ImageDraw.Draw(cone_layer)







    cd.polygon(



        [tuple(cone_tip), tuple(c1), tuple(c2)],



        fill=(*DARK_GREY, 10),



    )







    img.alpha_composite(cone_layer)







    d.line([tuple(cone_tip), tuple(c1)], fill=(*DARK_GREY, 10), width=1)



    d.line([tuple(cone_tip), tuple(c2)], fill=(*DARK_GREY, 10), width=1)







    # Intercept marker



    intercept = TRACK[min(idx + 62, len(TRACK) - 1)]



    ix, iy = intercept







    pulse = 0.55 + 0.45 * np.sin(phase * 5)







    d.ellipse(



        [ix - 22, iy - 22, ix + 22, iy + 22],



        outline=(*GREEN, int(120 + 80 * pulse)),



        width=2,



    )



    d.line((ix - 28, iy, ix + 28, iy), fill=(*GREEN, 120), width=1)



    d.line((ix, iy - 28, ix, iy + 28), fill=(*GREEN, 120), width=1)







    d.text(



        (ix + 34, iy - 12),



        "PREDICTED INTERCEPT",



        font=FONT_TINY,



        fill=(*GREEN, 180),



    )







    # Current object lock brackets



    ox, oy = obj



    jitter = 2.0 * np.sin(phase * 9)







    b = 34 + jitter



    c = 12







    for sx, sy in [(-1, -1), (1, -1), (-1, 1), (1, 1)]:



        x = ox + sx * b



        y = oy + sy * b







        d.line((x, y, x - sx * c, y), fill=(*CYAN2, 230), width=2)



        d.line((x, y, x, y - sy * c), fill=(*CYAN2, 230), width=2)







    d.ellipse([ox - 4, oy - 4, ox + 4, oy + 4], fill=(*CYAN2, 255))







    # Velocity vector



    next_p = TRACK[min(idx + 4, len(TRACK) - 1)]



    vv = next_p - obj



    vn = max(np.hypot(vv[0], vv[1]), 1)



    vv = vv / vn







    d.line(



        (ox, oy, ox + vv[0] * 52, oy + vv[1] * 52),



        fill=(*GREEN, 190),



        width=2,



    )







    # Right telemetry panel



    px0, py0 = 700, 90



    pw, ph = 220, 250







    d.rectangle([px0, py0, px0 + pw, py0 + ph], outline=(*CYAN, 100), width=1)



    d.line((px0, py0, px0 + 80, py0), fill=(*CYAN2, 190), width=3)







    d.text((px0 + 15, py0 + 15), "TRACK SOLUTION", font=FONT_SMALL, fill=(*CYAN2, 190))







    metrics = [



        ("RNG", f"{1200 - idx*1.7:06.1f} km"),



        ("VEL", f"{17.8 + 0.4*np.sin(phase):04.1f} km/s"),



        ("ETA", f"T+{max(0, 96 - i//2):03d}s"),



        ("ERR", f"{4.8 - 3.2*(i/FRAMES):04.2f} km"),



        ("FIT", f"{92 + 6*np.sin(phase*0.5)**2:04.1f}%"),



    ]







    for m, (k, v) in enumerate(metrics):



        yy = py0 + 54 + m * 34







        d.text((px0 + 16, yy), k, font=FONT_TINY, fill=(*CYAN, 150))



        d.text((px0 + 72, yy), v, font=FONT_SMALL, fill=(*CYAN2, 220))







    # Footer



    d.text(



        (32, H - 30),



        "PROPAGATING TARGET PATH • UNCERTAINTY CONE ACTIVE • INTERCEPT SOLUTION LOCKED",



        font=FONT_TINY,



        fill=(*CYAN, 130),



    )







    # HUD corners



    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)



    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)



    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)



    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)







    glow = img.filter(ImageFilter.GaussianBlur(2.0))



    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "target_prediction_hud",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/tracking/target_prediction_hud.gif


In [45]:
from pathlib import Path



import numpy as np



from PIL import Image, ImageDraw, ImageFilter, ImageFont







OUT_DIR = Path("media-site/animations/reconstruction")



OUT_DIR.mkdir(parents=True, exist_ok=True)







W, H = 960, 540



FPS = 24



FRAMES = 180







BG = (2, 7, 13, 255)







CYAN = (90, 240, 255)



CYAN2 = (180, 255, 255)



GREEN = (90, 255, 170)



YELLOW = (255, 220, 90)



WHITE = (245, 250, 255)







rng = np.random.default_rng(72)











def load_font(size):



    candidates = [



        "/System/Library/Fonts/Menlo.ttc",



        "/System/Library/Fonts/Supplemental/Arial Unicode.ttf",



        "/Library/Fonts/Arial.ttf",



        "DejaVuSansMono.ttf",



    ]



    for path in candidates:



        try:



            return ImageFont.truetype(path, size)



        except Exception:



            pass



    return ImageFont.load_default()











FONT = load_font(22)



FONT_SMALL = load_font(14)



FONT_TINY = load_font(11)











def smoothstep(t):



    t = np.clip(t, 0.0, 1.0)



    return t * t * (3 - 2 * t)











# -------------------------------------------------



# Procedural pseudo-3D object point cloud



# -------------------------------------------------



N = 180







theta = rng.uniform(0, 2 * np.pi, N)



phi = rng.uniform(-0.95, 0.95, N)







# irregular spheroid / artifact shell



rx = 190 * (1 + 0.14 * np.sin(3 * theta))



ry = 112 * (1 + 0.10 * np.cos(5 * theta))



rz = 85 * (1 + 0.18 * np.sin(4 * theta + 1.4))







x3 = rx * np.cos(theta) * np.sqrt(1 - phi**2)



y3 = ry * phi



z3 = rz * np.sin(theta) * np.sqrt(1 - phi**2)







POINTS3 = np.column_stack([x3, y3, z3])







# nearest-neighbour-ish edges



edges = []



for i in range(N):



    dist = np.linalg.norm(POINTS3 - POINTS3[i], axis=1)



    nn = np.argsort(dist)[1:4]



    for j in nn:



        if i < j:



            edges.append((i, j))







edges = edges[:260]











def project(points, phase):



    rot_y = phase * 0.35



    rot_x = 0.35 * np.sin(phase * 0.5)







    cy = np.cos(rot_y)



    sy = np.sin(rot_y)



    cxr = np.cos(rot_x)



    sxr = np.sin(rot_x)







    x = points[:, 0]



    y = points[:, 1]



    z = points[:, 2]







    # rotate y



    x2 = x * cy + z * sy



    z2 = -x * sy + z * cy







    # rotate x



    y3 = y * cxr - z2 * sxr



    z3 = y * sxr + z2 * cxr







    perspective = 1.0 / (1.0 + (z3 + 260) / 900)







    sx = W * 0.46 + x2 * perspective



    syy = H * 0.52 + y3 * perspective







    return np.column_stack([sx, syy, z3])











def render_frame(i):



    phase = 2 * np.pi * i / FRAMES







    img = Image.new("RGBA", (W, H), BG)



    d = ImageDraw.Draw(img)







    pts = project(POINTS3, phase)







    # phases



    point_a = smoothstep((i - 5) / 35)



    edge_a = smoothstep((i - 45) / 45)



    surface_a = smoothstep((i - 95) / 35)



    lock_a = smoothstep((i - 135) / 24)







    # Header



    d.text((32, 10), "WIREFRAME RECONSTRUCTION ENGINE", font=FONT_SMALL, fill=(*CYAN2, 190))



    d.line((28, 34, 260, 34), fill=(*CYAN, 170), width=2)



    d.line((260, 34, 282, 48), fill=(*CYAN, 170), width=2)



    d.line((282, 48, W - 42, 48), fill=(*CYAN, 75), width=1)







    # background calibration grid



    for x in range(80, W, 80):



        d.line((x, 72, x, H - 54), fill=(*CYAN, 12), width=1)



    for y in range(90, H - 50, 60):



        d.line((42, y, W - 42, y), fill=(*CYAN, 12), width=1)







    # surface hints: semi-transparent internal contour rings



    if surface_a > 0:



        cx0, cy0 = W * 0.46, H * 0.52







        for r, a0 in [(215, 22), (165, 28), (115, 35)]:



            wobble = 8 * np.sin(phase * 2 + r)



            d.ellipse(



                [



                    cx0 - r,



                    cy0 - r * 0.55 + wobble,



                    cx0 + r,



                    cy0 + r * 0.55 + wobble,



                ],



                outline=(*CYAN, int(a0 * surface_a)),



                width=1,



            )







    # edges



    max_edges = int(len(edges) * edge_a)







    for idx, (a, b) in enumerate(edges[:max_edges]):



        p1 = pts[a]



        p2 = pts[b]







        zmean = (p1[2] + p2[2]) * 0.5



        depth = np.clip((zmean + 120) / 260, 0.25, 1.0)







        alpha = int(30 + 90 * depth * edge_a)







        d.line(



            [tuple(p1[:2]), tuple(p2[:2])],



            fill=(*CYAN, alpha),



            width=1,



        )







    # points



    max_points = int(N * point_a)







    order = np.argsort(pts[:, 2])







    for idx in order[:max_points]:



        x, y, z = pts[idx]







        depth = np.clip((z + 120) / 260, 0.35, 1.0)



        pulse = 0.65 + 0.35 * np.sin(phase * 4 + idx)







        r = 2 + depth * 1.8



        alpha = int(95 + 120 * depth * pulse)







        d.ellipse(



            [x - r, y - r, x + r, y + r],



            fill=(*CYAN2, int(alpha * point_a)),



        )







    # scan ring / reconstruction sweep



    sweep_r = 70 + (i * 3.2) % 260



    cx0, cy0 = W * 0.46, H * 0.52







    d.ellipse(



        [



            cx0 - sweep_r,



            cy0 - sweep_r * 0.55,



            cx0 + sweep_r,



            cy0 + sweep_r * 0.55,



        ],



        outline=(*GREEN, 70),



        width=1,



    )







    # final lock brackets



    if lock_a > 0:



        bx0, by0 = W * 0.46 - 245, H * 0.52 - 165



        bx1, by1 = W * 0.46 + 245, H * 0.52 + 165



        c = 40







        col = CYAN2



        a = int(210 * lock_a)







        # TL



        d.line((bx0, by0, bx0 + c, by0), fill=(*col, a), width=3)



        d.line((bx0, by0, bx0, by0 + c), fill=(*col, a), width=3)



        # TR



        d.line((bx1, by0, bx1 - c, by0), fill=(*col, a), width=3)



        d.line((bx1, by0, bx1, by0 + c), fill=(*col, a), width=3)



        # BL



        d.line((bx0, by1, bx0 + c, by1), fill=(*col, a), width=3)



        d.line((bx0, by1, bx0, by1 - c), fill=(*col, a), width=3)



        # BR



        d.line((bx1, by1, bx1 - c, by1), fill=(*col, a), width=3)



        d.line((bx1, by1, bx1, by1 - c), fill=(*col, a), width=3)







    # Right status panel



    px, py = 690, 95



    pw, ph = 230, 280







    d.rectangle([px, py, px + pw, py + ph], outline=(*CYAN, 100), width=1)



    d.line((px, py, px + 85, py), fill=(*CYAN2, 190), width=3)







    d.text((px + 16, py + 16), "RECONSTRUCTION", font=FONT_SMALL, fill=(*CYAN2, 190))







    metrics = [



        ("POINTS", f"{max_points:03d}/{N:03d}"),



        ("EDGES", f"{max_edges:03d}/{len(edges):03d}"),



        ("SURFACE", f"{int(surface_a*100):03d}%"),



        ("LOCK", f"{int(lock_a*100):03d}%"),



        ("DRIFT", f"{0.003*np.sin(phase):+.4f}"),



    ]







    for m, (k, v) in enumerate(metrics):



        yy = py + 58 + m * 38



        d.text((px + 18, yy), k, font=FONT_TINY, fill=(*CYAN, 150))



        d.text((px + 100, yy), v, font=FONT_SMALL, fill=(*CYAN2, 220))







        for t in range(5):



            a = int(35 + 90 * np.sin(phase * 4 + m + t) ** 2)



            tx = px + 190 + t * 6



            d.line((tx, yy + 8, tx + 3, yy + 8), fill=(*CYAN, a), width=1)







    # footer



    d.text(



        (32, H - 30),



        "POINT CLOUD ALIGNMENT • TOPOLOGY SOLVER ACTIVE • SURFACE RECONSTRUCTION LOCK",



        font=FONT_TINY,



        fill=(*CYAN, 130),



    )







    # HUD corners



    d.line((0, 0, 44, 0), fill=(*CYAN, 190), width=2)



    d.line((0, 0, 0, 22), fill=(*CYAN, 190), width=2)



    d.line((W - 44, 0, W, 0), fill=(*CYAN, 190), width=2)



    d.line((W - 1, 0, W - 1, 22), fill=(*CYAN, 190), width=2)







    glow = img.filter(ImageFilter.GaussianBlur(2.0))



    final = Image.new("RGBA", (W, H), BG)



    final.alpha_composite(glow)



    final.alpha_composite(img)







    return final











frames = [render_frame(i) for i in range(FRAMES)]







import numpy as np

from vizlib.animation_export import export_animation



saved = export_animation(

    [np.asarray(frame.convert("RGBA")) for frame in frames],

    OUT_DIR,

    "wireframe_reconstruction",

    "webm",

    FPS,

)

print(f"Saved: {saved}")



Saved: animations/reconstruction/wireframe_reconstruction.gif
